# Task B — Severity Prediction: Complete Pipeline
## FAERS Drug Interactions Risk Project

Este notebook ejecuta **todo el pipeline de Task B** de principio a fin:

1. Verifica dependencias y genera las reglas filtradas de Task A si no existen
2. Ingesta los datos crudos del ZIP de FAERS
3. Construye la feature matrix de 71 columnas
4. Entrena y evalua 5 modelos: Logistic Regression, Random Forest, XGBoost (GPU), CatBoost (GPU), Voting Ensemble
5. Optimizacion de umbral de clasificacion por F1 (severe) en validation set
6. Genera plots de evaluacion embebidos en el notebook
7. Optuna hyperparameter tuning (Bayesian TPE): XGBoost 100 trials + CatBoost 50 trials (GPU)
8. Genera plots de evaluacion de modelos tunados

**Correr desde la raiz del proyecto** ()

In [ ]:
%matplotlib inline
import os, sys, subprocess, glob, warnings, time
warnings.filterwarnings('ignore')

# ── Ensure we are at project root ─────────────────────────────────────────────
notebook_dir = os.path.abspath('')
if not os.path.isdir(os.path.join(notebook_dir, 'taskA')):
    parent = os.path.dirname(notebook_dir)
    if os.path.isdir(os.path.join(parent, 'taskA')):
        os.chdir(parent)

print('Working directory:', os.getcwd())
assert os.path.isdir('taskA'), 'ERROR: Run this notebook from the DrugInteractionsRisks/ root folder!'
print('Project root confirmed ✓')

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc, f1_score,
    precision_recall_curve, average_precision_score,
)

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
TEST_SIZE    = 0.20
VAL_FRAC     = 0.10   # fraction of training set for threshold tuning

# ── Paths ─────────────────────────────────────────────────────────────────────
ZIP_PATH        = os.path.join('taskB', '9 json files - no tocar.zip')
CONSOLIDATED    = 'consolidated_data.parquet'
ENCODED_PARQUET = os.path.join('taskA', 'active_substances_encoded.parquet')
RULES_RAW_DIR   = os.path.join('taskA', 'association_rules')
FILTERED_DIR    = os.path.join('taskA', 'filtered_association_rules')
FILTERED_GLOB   = os.path.join(FILTERED_DIR, '*.csv')
FEATURES_PATH   = os.path.join('taskB', 'task_b_features.parquet')
OUTPUT_DIR      = os.path.join('taskB', 'TaskBPlots')
EVAL_CSV        = os.path.join('taskB', 'task_b_evaluation.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Paths configured ✓')

---
## FASE 0 — Dependency Check
Verifica qué ficheros existen y cuáles hay que generar.

In [ ]:
def check(label, path):
    exists = os.path.exists(path)
    icon   = '✓' if exists else '✗ MISSING'
    print(f'  {icon:<10} {label:<45} {path}')
    return exists

print('=== DEPENDENCY CHECK ===')
has_encoded  = check('Encoded parquet (Task A)',  ENCODED_PARQUET)
has_filtered = check('Filtered rules dir',         FILTERED_DIR)
has_consol   = check('consolidated_data.parquet',  CONSOLIDATED)
has_zip      = check('FAERS ZIP file',             ZIP_PATH)
has_features = check('task_b_features.parquet',    FEATURES_PATH)

print()
if not has_encoded:
    raise FileNotFoundError(
        'CRITICAL: active_substances_encoded.parquet missing. '
        'This file must come from the git repo — it cannot be regenerated without consolidated_data.parquet.'
    )
if not has_zip and not has_consol:
    raise FileNotFoundError(
        f'CRITICAL: Need either {ZIP_PATH} or {CONSOLIDATED}. '
        'Place the ZIP at the path above.'
    )
print('Minimum requirements satisfied ✓')

---
## FASE 1 — Generar Filtered Association Rules (Task A → Task B bridge)

Las reglas filtradas de Task A son necesarias para las 4 features de interacción en Task B.  
Los encoded parquets YA están en el repo, así que esto corre **sin necesitar el ZIP**.

In [ ]:
filtered_files = glob.glob(FILTERED_GLOB)

if filtered_files:
    print(f'Filtered rules already exist ({len(filtered_files)} file(s)):')
    for f in filtered_files:
        df_tmp = pd.read_csv(f)
        print(f'  {os.path.basename(f)} — {len(df_tmp)} rules, max_lift={df_tmp["lift"].max():.2f}')
else:
    print('Filtered rules not found. Generating from encoded parquet...')

    # ── Step 1: Apriori on existing encoded parquet ────────────────────────────
    apriori_cmd = [
        sys.executable, 'taskA/4A_APriori.py',
        '--file_path', ENCODED_PARQUET,
        '--min_support', '0.0025',
    ]
    print('\nRunning Apriori...')
    r1 = subprocess.run(apriori_cmd, capture_output=True, text=True)
    if r1.returncode != 0:
        print('STDERR:', r1.stderr)
        raise RuntimeError('4A_APriori.py failed — see STDERR above')
    print(r1.stdout[-2000:])   # last 2000 chars to avoid flooding

    # ── Step 2: Filter ────────────────────────────────────────────────────────
    apriori_csv = os.path.join(
        RULES_RAW_DIR, 'association_rules_APRIORI_active_substances_0_0025.csv'
    )
    if not os.path.exists(apriori_csv):
        raise FileNotFoundError(
            f'Expected Apriori CSV not found at {apriori_csv}. '
            'Check 4A_APriori.py output above.'
        )

    filter_cmd = [
        sys.executable, 'taskA/6A_FilterTrueInteractions.py',
        '--input', apriori_csv,
    ]
    print('\nRunning filter...')
    r2 = subprocess.run(filter_cmd, capture_output=True, text=True)
    if r2.returncode != 0:
        print('STDERR:', r2.stderr)
        raise RuntimeError('6A_FilterTrueInteractions.py failed — see STDERR above')
    print(r2.stdout)

    filtered_files = glob.glob(FILTERED_GLOB)
    if not filtered_files:
        raise RuntimeError(
            'Filtering ran but no CSVs found in taskA/filtered_association_rules/. '
            'Possibly no rules survived the filter — check noise_terms in 6A script.'
        )

print(f'\nFiltered rules ready: {len(filtered_files)} file(s) ✓')

---
## FASE 2 — Data Ingestion

Lee el ZIP con los 9 JSON de FAERS y genera `consolidated_data.parquet`.  
Si el parquet ya existe, se salta este paso.

In [ ]:
if os.path.exists(CONSOLIDATED):
    _tmp = pd.read_parquet(CONSOLIDATED)
    print(f'consolidated_data.parquet already exists: {len(_tmp):,} rows × {len(_tmp.columns)} columns ✓')
    del _tmp
else:
    print(f'Running ingestion from {ZIP_PATH}...')
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, 'taskB/1B_dataIngestion.py', ZIP_PATH],
        capture_output=True, text=True
    )
    print(r.stdout)
    if r.returncode != 0:
        print('STDERR:', r.stderr)
        raise RuntimeError('1B_dataIngestion.py failed')
    if not os.path.exists(CONSOLIDATED):
        raise FileNotFoundError('Ingestion ran but consolidated_data.parquet was not created')
    print(f'Ingestion done in {time.time()-t0:.1f}s ✓')

---
## FASE 3 — Feature Engineering

Construye la feature matrix a partir de `consolidated_data.parquet` y las reglas filtradas.  
Si `task_b_features.parquet` ya existe, se salta.

In [ ]:
if os.path.exists(FEATURES_PATH):
    df_feat = pd.read_parquet(FEATURES_PATH)
    print(f'task_b_features.parquet already exists: {df_feat.shape} ✓')
else:
    print('Running feature engineering (2B_preprocessing.py)...')
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, 'taskB/2B_preprocessing.py'],
        capture_output=True, text=True
    )
    print(r.stdout)
    if r.returncode != 0:
        print('STDERR:', r.stderr)
        raise RuntimeError('2B_preprocessing.py failed')
    if not os.path.exists(FEATURES_PATH):
        raise FileNotFoundError('Preprocessing ran but task_b_features.parquet was not created')
    df_feat = pd.read_parquet(FEATURES_PATH)
    print(f'Feature engineering done in {time.time()-t0:.1f}s ✓')

print(f'\nFeature matrix: {df_feat.shape[0]:,} rows × {df_feat.shape[1]} columns')
print('Columns:', list(df_feat.columns))

---
## FASE 4 — Exploratory Analysis of Feature Matrix

In [ ]:
# ── Class balance ─────────────────────────────────────────────────────────────
counts = df_feat['is_severe_outcome'].value_counts()
n0     = counts.get(0, 0)
n1     = counts.get(1, 0)
total  = len(df_feat)

print('=== CLASS BALANCE ===')
print(f'  Non-Severe (0): {n0:>7,}  ({n0/total*100:.1f}%)')
print(f'  Severe     (1): {n1:>7,}  ({n1/total*100:.1f}%)')
print(f'  Total:          {total:>7,}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot 1: class balance bar
axes[0].bar(['Non-Severe (0)', 'Severe (1)'], [n0, n1],
            color=sns.color_palette('Set2', 2))
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_ylabel('Reports')
for i, v in enumerate([n0, n1]):
    axes[0].text(i, v + 500, f'{v:,}\n({v/total*100:.1f}%)', ha='center', fontsize=9)

# Plot 2: age distribution by class
if 'patientonsetage' in df_feat.columns:
    for label, grp in df_feat.groupby('is_severe_outcome'):
        axes[1].hist(grp['patientonsetage'].dropna(), bins=40, alpha=0.6,
                     label=f'{"Severe" if label==1 else "Non-Severe"} (n={len(grp):,})')
    axes[1].set_title('Age Distribution by Severity', fontweight='bold')
    axes[1].set_xlabel('Patient Age (years)')
    axes[1].legend(fontsize=8)

# Plot 3: polypharmacy index distribution by class
if 'num_drugs_taken' in df_feat.columns:
    for label, grp in df_feat.groupby('is_severe_outcome'):
        axes[2].hist(grp['num_drugs_taken'], bins=range(0, 20), alpha=0.6,
                     label=f'{"Severe" if label==1 else "Non-Severe"}')
    axes[2].set_title('Polypharmacy Index by Severity', fontweight='bold')
    axes[2].set_xlabel('Unique drugs taken')
    axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eda_distributions.png'), dpi=150)
plt.show()

In [ ]:
# ── Interaction features health check ─────────────────────────────────────────
print('=== INTERACTION FEATURES (Task A → Task B bridge) ===')
interaction_cols = [
    'has_drug_drug_interaction', 'has_drug_reaction_rule',
    'num_matching_rules', 'max_interaction_lift'
]
all_zeros = True
for col in interaction_cols:
    if col not in df_feat.columns:
        print(f'  MISSING column: {col}')
        continue
    v = df_feat[col]
    if col.startswith('has_'):
        n = int(v.sum())
        pct = n / total * 100
        print(f'  {col}: {n:,} reports flagged ({pct:.1f}%)')
        if n > 0:
            all_zeros = False
    else:
        nonzero = int((v > 0).sum())
        print(f'  {col}: mean={v.mean():.3f}, max={v.max():.2f}, nonzero={nonzero:,}')

if all_zeros:
    print()
    print('⚠ WARNING: All interaction features are zero!')
    print('  The filtered association rules were not used by 2B_preprocessing.py.')
    print('  Delete task_b_features.parquet and re-run FASE 3 after FASE 1 completes.')
else:
    print('\nInteraction features look good ✓')

In [ ]:
# ── Missing values summary ─────────────────────────────────────────────────────
print('=== MISSING VALUES ===')
missing = df_feat.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    for col, n in missing.items():
        print(f'  {col}: {n:,} NaN ({n/total*100:.1f}%)')
else:
    print('  No missing values (NaN in patientonsetage is expected and handled per-model)')

---
## FASE 5 - Data Split

Split strategy: **70% train / 10% val / 20% test** (stratified, random_state=42).

- Val set used **only** for threshold tuning (grid search F1-severe) -- never for model selection.
- Applied inside 3B_modeling.py to avoid leakage from the notebook context.

---
## FASE 6 - Model Training

Trains 5 classifiers via 3B_modeling.py:
- Logistic Regression (linear baseline)
- Random Forest (300 trees)
- XGBoost (GPU, device=cuda)
- CatBoost (GPU)
- Soft Voting Ensemble (RF + XGBoost + CatBoost)

Each model uses an sklearn.Pipeline with per-model imputation and optional scaling.
Class imbalance handled via class_weight and scale_pos_weight.
Threshold tuning on val set (grid 0.10-0.70, maximise F1-severe).

In [ ]:
# Run 3B_modeling.py
import subprocess, sys
r = subprocess.run(
    [sys.executable, "taskB/3B_modeling.py"],
    capture_output=True, text=True, encoding="utf-8", errors="replace"
)
print(r.stdout[-5000:] if len(r.stdout) > 5000 else r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr[-2000:])
    raise RuntimeError("3B_modeling.py failed")

---
## FASE 7 - Evaluation Plots & Results Table

In [ ]:
# Display evaluation plots
from IPython.display import Image, display
import os

for img_path, caption in [
    ("taskB/TaskBPlots/roc_curves.png",             "ROC Curves -- all 5 models"),
    ("taskB/TaskBPlots/confusion_matrices.png",     "Confusion Matrices (normalized)"),
    ("taskB/TaskBPlots/pr_curves.png",              "Precision-Recall Curves"),
    ("taskB/TaskBPlots/feature_importance_xgb.png", "XGBoost Feature Importance"),
    ("taskB/TaskBPlots/feature_importance_rf.png",  "Random Forest Feature Importance"),
    ("taskB/TaskBPlots/lr_coefficients.png",        "Logistic Regression Coefficients"),
]:
    if os.path.exists(img_path):
        print(f"--- {caption} ---")
        display(Image(img_path))
    else:
        print(f"[missing] {img_path}")

In [ ]:
# Results table
import pandas as pd
from IPython.display import display

df = pd.read_csv("taskB/task_b_evaluation.csv")
print("=== BASELINE MODEL RESULTS ===")
display(df.style.highlight_max(
    subset=["AUC-ROC", "Avg Precision", "F1 (severe=1)", "Recall (severe)"],
    color="#d4edda"
))

---
## FASE 8.5 — Optuna Hyperparameter Tuning

Optimizacion automatica de hiperparametros con **Optuna** (Bayesian TPE sampler).

| Model | Trials | Early stopping | Search space |
|-------|--------|----------------|--------------|
| XGBoost (GPU) | 100 | n_estimators=3000, patience=50 | lr, depth, subsample, colsample, mcw, gamma, reg_alpha, reg_lambda |
| CatBoost (GPU) | 50 | iterations=2000, patience=50 | lr, depth, l2_leaf_reg, bagging_temp, random_strength, min_data_in_leaf |

Despues del tuning: re-entrena con mejores params, evalua en test, construye ensemble tunado.

In [ ]:
# ── Run Optuna tuning ─────────────────────────────────────────────────────────
import subprocess, sys
r = subprocess.run(
    [sys.executable, "taskB/4B_optuna_tuning.py"],
    capture_output=True, text=True, encoding="utf-8", errors="replace"
)
print(r.stdout[-4000:] if len(r.stdout) > 4000 else r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr[-2000:])
    raise RuntimeError("4B_optuna_tuning.py failed")

In [ ]:
# Load and display tuned results
import json, pandas as pd
from IPython.display import display

tuned_df = pd.read_csv("taskB/task_b_tuned_evaluation.csv")
print()
print("=== TUNED MODEL RESULTS ===")
display(tuned_df.style.highlight_max(
    subset=["AUC-ROC", "Avg Precision", "F1 (severe=1)", "Recall (severe)"],
    color="#d4edda"
))

# Baseline vs tuned delta
if os.path.exists("taskB/task_b_evaluation.csv"):
    base_df = pd.read_csv("taskB/task_b_evaluation.csv")
    print()
    print("=== BASELINE vs TUNED (AUC-ROC delta) ===")
    pairs = [("XGBoost (GPU)", "XGBoost Tuned (GPU)"),
             ("CatBoost (GPU)", "CatBoost Tuned (GPU)"),
             ("Voting Ensemble", "Tuned Ensemble")]
    for bname, tname in pairs:
        b = base_df[base_df["Model"]==bname]["AUC-ROC"].values
        t = tuned_df[tuned_df["Model"]==tname]["AUC-ROC"].values
        if len(b) and len(t):
            delta = t[0] - b[0]
            sign = "+" if delta >= 0 else ""
            print(f"  {bname:<22} {b[0]:.4f} -> {tname:<28} {t[0]:.4f}  ({sign}{delta:.4f})")

# Best params
with open("taskB/optuna_best_params.json") as fp:
    best_params = json.load(fp)
print()
print("=== BEST XGBoost PARAMS ===")
print(json.dumps(best_params["xgboost"], indent=2))
print()
print("=== BEST CatBoost PARAMS ===")
print(json.dumps(best_params["catboost"], indent=2))

In [ ]:
# ── Display optimization history and ROC curves ────────────────────────────
from IPython.display import Image, display

for img_path, caption in [
    ("taskB/TaskBPlots/optuna_history_xgb.png",     "XGBoost Optuna History"),
    ("taskB/TaskBPlots/optuna_history_catboost.png", "CatBoost Optuna History"),
    ("taskB/TaskBPlots/roc_curves_tuned.png",        "ROC Curves — Tuned Models"),
    ("taskB/TaskBPlots/feature_importance_xgb_tuned.png", "XGBoost Tuned Feature Importance"),
]:
    if os.path.exists(img_path):
        print(f"--- {caption} ---")
        display(Image(img_path))

---
## FASE 9 — Conclusiones

**Lo que hemos construido:**
- Pipeline KDD completo: ~108k reportes FAERS → feature matrix de 25 columnas → 3 clasificadores evaluados
- 4 grupos de features: demographics, polypharmacy index, top-15 substance flags, interaction risk (from Task A)
- Evaluación correcta para clases desbalanceadas: AUC-ROC, Average Precision, y F1 — no accuracy
- Optimización de umbral de clasificación en validation set separado (sin data leakage)
- Sin data leakage: imputation + scaling encapsulados dentro de cada `sklearn.Pipeline`

**El puente Task A → Task B:**  
`has_drug_reaction_rule` y `max_interaction_lift` son las features que conectan directamente el conocimiento farmacológico descubierto por association rules con el clasificador de severidad.

**Modelos:**
- **Logistic Regression**: baseline interpretable, coeficientes directamente legibles como log-odds
- **Random Forest**: ensemble no-lineal, feature importance por reducción de impureza Gini
- **Gradient Boosting (HGBC)**: boosting por histogramas, maneja NaN nativamente, mejor trade-off bias-varianza

**Limitaciones:**
- FAERS tiene selection bias: solo reportes voluntarios de eventos adversos
- Association rules capturan correlación, no causalidad
- ~40% NaN en edad, manejado por imputation con mediana dentro de cada pipeline
- AUC ~0.70 refleja el ruido inherente de FAERS — no todas las variables de confusión están disponibles

In [ ]:
print('=== NOTEBOOK COMPLETE ===')
print(f'\nPlots saved to {OUTPUT_DIR}/')
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f'  {f}')
print(f'\nEvaluation table: {EVAL_CSV}')